<a href="https://colab.research.google.com/github/kougiyous-arch/lecture-ai-engineering-2026/blob/lec02/lora_format_tuning_assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する

架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。

**この演習で確かめること**

1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。
2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。
3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.1（2026-10-18）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 4,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 4,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## 2. 架空 FAQ と教師データの生成

**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。

- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。
- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。

**データの分け方**

- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。
- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。
- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。

**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。

In [4]:

import json
import re
from google.colab import files

# Codespacesで生成したJSONLをアップロード
files.upload()

with open("glossary_sft.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

# JSONLから用語と定義を抽出
GLOSSARY = {}

for record in records:
    q = record["messages"][0]["content"]
    a = record["messages"][1]["content"]

    m = re.fullmatch(r"「(.+?)」とは何ですか？", q)
    if m:
        GLOSSARY[m.group(1)] = a

print("学習データ:", len(records))
print("用語数:", len(GLOSSARY))

assert len(records) == 116
assert len(GLOSSARY) == 58

# 評価問題を自分で作成
# 区分、正解の用語、質問
EVAL_DATA = [
    # 用語集にある語：学習と同じ形式
    ("収録語", "LoRA",
     "「LoRA」とは何ですか？"),

    ("収録語", "過学習",
     "「過学習」とは何ですか？"),

    ("収録語", "ハルシネーション",
     "「ハルシネーション」とは何ですか？"),

    ("収録語", "APIキー",
     "「APIキー」とは何ですか？"),

    ("収録語", "コンテキストウィンドウ",
     "「コンテキストウィンドウ」とは何ですか？"),

    # 言い回しを変えた質問：学習にはない表現
    ("言い換え", "LoRA",
     "元の重みを固定し、低ランク行列だけを学習する手法は何ですか？"),

    ("言い換え", "過学習",
     "訓練データには適合するのに未知のデータで性能が下がる状態を何と呼びますか？"),

    ("言い換え", "ハルシネーション",
     "LLMが事実と異なる情報をもっともらしく生成する現象は何ですか？"),

    ("言い換え", "APIキー",
     "APIを利用する人を識別する秘密の文字列を何と呼びますか？"),

    ("言い換え", "コンテキストウィンドウ",
     "LLMが1回で扱える入力と出力の合計トークン数の上限を何と呼びますか？"),

    # 用語集にない語
    ("未登録語", "BLEUスコア",
     "「BLEUスコア」とは何ですか？"),

    ("未登録語", "バッチ正規化",
     "「バッチ正規化」とは何ですか？"),

    ("未登録語", "モデル蒸留",
     "「モデル蒸留」とは何ですか？"),
]

test = []
gold = {}

for kind, term, question in EVAL_DATA:
    if kind == "収録語":
        assert term in GLOSSARY
        expected = GLOSSARY[term]

    elif kind == "言い換え":
        assert term in GLOSSARY
        expected = term

    else:
        # 用語集の見出し・定義文にないことを確認
        assert term not in GLOSSARY
        assert not any(
            term in definition
            for definition in GLOSSARY.values()
        )
        expected = None

    test.append({
        "question": question,
        "kind": kind,
        "term": term
    })

    gold[question] = {
        "kind": kind,
        "term": term,
        "expected": expected
    }

print("評価問題:", len(test))

for ex in test:
    print(
        ex["kind"],
        ex["question"],
        "正解:", gold[ex["question"]]["expected"]
    )

# 評価問題と正解を保存
_ = (WORK / "test.jsonl").write_text(
    "\n".join(
        json.dumps(r, ensure_ascii=False)
        for r in test
    ),
    encoding="utf-8"
)

_ = (WORK / "gold.json").write_text(
    json.dumps(gold, ensure_ascii=False, indent=2),
    encoding="utf-8"
)


Saving glossary_sft.jsonl to glossary_sft.jsonl
学習データ: 116
用語数: 58
評価問題: 13
収録語 「LoRA」とは何ですか？ 正解: 事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
収録語 「過学習」とは何ですか？ 正解: 学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
収録語 「ハルシネーション」とは何ですか？ 正解: LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
収録語 「APIキー」とは何ですか？ 正解: APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。
収録語 「コンテキストウィンドウ」とは何ですか？ 正解: LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。
言い換え 元の重みを固定し、低ランク行列だけを学習する手法は何ですか？ 正解: LoRA
言い換え 訓練データには適合するのに未知のデータで性能が下がる状態を何と呼びますか？ 正解: 過学習
言い換え LLMが事実と異なる情報をもっともらしく生成する現象は何ですか？ 正解: ハルシネーション
言い換え APIを利用する人を識別する秘密の文字列を何と呼びますか？ 正解: APIキー
言い換え LLMが1回で扱える入力と出力の合計トークン数の上限を何と呼びますか？ 正解: コンテキストウィンドウ
未登録語 「BLEUスコア」とは何ですか？ 正解: None
未登録語 「バッチ正規化

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def build_messages(example):
    return [
        {"role": "user", "content": example["question"]}
    ]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:

import pandas as pd
pd.set_option("display.max_colwidth", None)

def evaluate(model, label):
    rows = []
    t0 = time.time()

    for ex in test:
        raw = generate(model, ex)
        correct = gold[ex["question"]]

        rows.append({
            "question": ex["question"],
            "kind": ex["kind"],
            "term": ex["term"],
            "expected": correct["expected"],
            "raw": raw
        })

    summary = {
        "label": label,
        "n": len(rows),
        "seconds": round(time.time() - t0, 1)
    }

    (WORK / f"eval_{label}.json").write_text(
        json.dumps(
            {"summary": summary, "rows": rows},
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

# 学習前評価
before_summary, before_rows = evaluate(model, "before")

display(pd.DataFrame([
    {
        "区分": r["kind"],
        "質問": r["question"],
        "正解": r["expected"],
        "学習前の回答": r["raw"]
    }
    for r in before_rows
]))


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 13, "seconds": 89.0}


,区分,質問,正解,学習前の回答
0,収録語,「LoRA」とは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、
1,収録語,「過学習」とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに
2,収録語,「ハルシネーション」とは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,**ハルシネーション**（Hārinēshō）は、日本語で「**ハルシネーション**」です。\n\nこの言葉は、主に**物理学**（特に光の現象）や**音楽学**（特に弦楽器の音）において使われます。\n\n* **物理学の定義**: 光の波が媒質（空気、水など）を通過する際、波の振幅が媒質の密度や
3,収録語,「APIキー」とは何ですか？,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。,**API キー**（Application API Key）とは、アプリケーション（アプリやサービス）が、外部のサービス（API）にアクセスするための**認証情報**です。\n\n簡単に言うと、「あなたのアプリが誰かにデータを送るためのパスワード」や「アクセス権限の証明」のことです。\n\n### 主な特徴と仕組み\n\n1. **認証の必須条件**\n 多くの API サービス（例：Google API、Facebook API、AWS API など
4,収録語,「コンテキストウィンドウ」とは何ですか？,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,**コンテキストウィンドウ**（Context Window）とは、大規模なテキストベースの AI（特に大規模言語モデル）において、**「入力されたテキストの全体」を一つの大きな情報セットとして認識・処理する仕組み**です。\n\n従来のテキスト処理では、単語単位で処理する一方、コンテキストウィンドウでは、**「文脈（前後の文脈）」を考慮して、単語の意味や文脈を同時に理解・生成する
5,言い換え,元の重みを固定し、低ランク行列だけを学習する手法は何ですか？,LoRA,元の重みを固定し、低ランク行列だけを学習する手法は、**低ランク学習（Low-Rank Learning）**、あるいは**低ランク分解（Low-Rank Decomposition）**と呼ばれます。\n\nこの手法は、データやモデルの構造を「低ランク」であると仮定して、その構造を最小二乗法などで最適化することで、より効率的に学習できるアプローチです。\n\n具体的には、以下の 2 つの主要なアプローチがあります。
6,言い換え,訓練データには適合するのに未知のデータで性能が下がる状態を何と呼びますか？,過学習,その状態を**欠損値（Missing Value）**と呼びます。\n\nこれは、訓練データに含まれていない（または入力された値が不明）部分のことを指します。この欠損値が訓練データに直結している場合、モデルは欠損値を無視して学習し、結果として**欠損率（Missing Rate）**が高くなることで、最終的な予測精度が低下します。\n\n欠損値の処理には、**欠損値
7,言い換え,LLMが事実と異なる情報をもっともらしく生成する現象は何ですか？,ハルシネーション,LLM（大規模言語モデル）が事実と異なる情報を「もっともらしく」生成する現象は、主に**「誤解された事実（Misinformation）」**や**「誤った情報（Misleading Information）」**と呼ばれます。\n\nこれは、モデルが学習した知識に基づいて、特定の文脈や目的で歪曲された情報を生成する現象です。その特徴は以下の通りです。\n\n1. **事実と異なる情報**：モデルが
8,言い換え,APIを利用する人を識別する秘密の文字列を何と呼びますか？,APIキー,API を利用する人を識別する秘密の文字列を**API キー**（あるいは API Key）と呼びます。\n\nこれは、特定のサービス（例：Google API、AWS API、Slack API など）にアクセスするための一意のパスワードや認証コードのことです。\n\n* **API キー**：サービスにアクセスするための認証情報。\n* **API キー**：サービスにアクセスするための認証情報。\n\nこの情報は、ユーザーのパスワード
9,言い換え,LLMが1回で扱える入力と出力の合計トークン数の上限を何と呼びますか？,コンテキストウィンドウ,LLM（大規模言語モデル）が 1 回で処理できる入力と出力の合計トークン数の上限を**「最大トークン数（Maximum Token Length）」**と呼びます。\n\nこの値は、モデルの設計パラメータ（例えば、GPT-3.5 の 4096 個のトークン、GPT-4 の 8192 個など）によって異なります。また、この上限は


## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(records)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 1,597,440 || all params: 753,990,464 || trainable%: 0.2119


Step,Training Loss
10,3.265619
20,2.214392
30,1.686354
40,0.971207
50,0.550307
60,0.344415
70,0.184758
80,0.153981
90,0.134699
100,0.113394


学習時間: 682 秒 | ステップ数: 300 | エポック: 20.0


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [8]:

# 学習後評価
after_summary, after_rows = evaluate(tuned, "after")

# 評価時間・問題数の比較
compare = pd.DataFrame(
    [before_summary, after_summary]
).set_index("label")

display(compare)

# 学習前後の回答比較
detail = pd.DataFrame([
    {
        "区分": b["kind"],
        "質問": b["question"],
        "正解": (
            b["expected"]
            if b["expected"] is not None
            else "用語集に記載なし"
        ),
        "学習前": b["raw"],
        "学習後": a["raw"]
    }
    for b, a in zip(before_rows, after_rows)
])

display(detail)


{"label": "after", "n": 13, "seconds": 46.6}


,n,seconds
label,,
before,13,89.0
after,13,46.6


,区分,質問,正解,学習前,学習後
0,収録語,「LoRA」とは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
1,収録語,「過学習」とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
2,収録語,「ハルシネーション」とは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,**ハルシネーション**（Hārinēshō）は、日本語で「**ハルシネーション**」です。\n\nこの言葉は、主に**物理学**（特に光の現象）や**音楽学**（特に弦楽器の音）において使われます。\n\n* **物理学の定義**: 光の波が媒質（空気、水など）を通過する際、波の振幅が媒質の密度や,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
3,収録語,「APIキー」とは何ですか？,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。,**API キー**（Application API Key）とは、アプリケーション（アプリやサービス）が、外部のサービス（API）にアクセスするための**認証情報**です。\n\n簡単に言うと、「あなたのアプリが誰かにデータを送るためのパスワード」や「アクセス権限の証明」のことです。\n\n### 主な特徴と仕組み\n\n1. **認証の必須条件**\n 多くの API サービス（例：Google API、Facebook API、AWS API など,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。
4,収録語,「コンテキストウィンドウ」とは何ですか？,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,**コンテキストウィンドウ**（Context Window）とは、大規模なテキストベースの AI（特に大規模言語モデル）において、**「入力されたテキストの全体」を一つの大きな情報セットとして認識・処理する仕組み**です。\n\n従来のテキスト処理では、単語単位で処理する一方、コンテキストウィンドウでは、**「文脈（前後の文脈）」を考慮して、単語の意味や文脈を同時に理解・生成する,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。
5,言い換え,元の重みを固定し、低ランク行列だけを学習する手法は何ですか？,LoRA,元の重みを固定し、低ランク行列だけを学習する手法は、**低ランク学習（Low-Rank Learning）**、あるいは**低ランク分解（Low-Rank Decomposition）**と呼ばれます。\n\nこの手法は、データやモデルの構造を「低ランク」であると仮定して、その構造を最小二乗法などで最適化することで、より効率的に学習できるアプローチです。\n\n具体的には、以下の 2 つの主要なアプローチがあります。,PNN（Proximal Neuron）やLoRA。LoRAでは重みを交換しないので学習が安定し、PNNでは初期の好な行列を共有するので学習も安定する。
6,言い換え,訓練データには適合するのに未知のデータで性能が下がる状態を何と呼びますか？,過学習,その状態を**欠損値（Missing Value）**と呼びます。\n\nこれは、訓練データに含まれていない（または入力された値が不明）部分のことを指します。この欠損値が訓練データに直結している場合、モデルは欠損値を無視して学習し、結果として**欠損率（Missing Rate）**が高くなることで、最終的な予測精度が低下します。\n\n欠損値の処理には、**欠損値,未知の入力に直面し、モデルの挙動や出力が安定しないようにデータに付けている手法の総称。Pydantic AIでは **LoRAアダプタの lora LoRA居士**。モデルが知らない入力に直面し、モデルが知らないことにもっとする精度落ちを補うための保険の型。本講座第2回の演習ではLoRAアダプタを扱う。
7,言い換え,LLMが事実と異なる情報をもっともらしく生成する現象は何ですか？,ハルシネーション,LLM（大規模言語モデル）が事実と異なる情報を「もっともらしく」生成する現象は、主に**「誤解された事実（Misinformation）」**や**「誤った情報（Misleading Information）」**と呼ばれます。\n\nこれは、モデルが学習した知識に基づいて、特定の文脈や目的で歪曲された情報を生成する現象です。その特徴は以下の通りです。\n\n1. **事実と異なる情報**：モデルが,質問に存在する事実と異なる情報を、モデルが知らないことや、訓練データにないことや、最新情報や組織に固有の情報や、特定の業界の常識や教条に似たような情報を、もっともらしく生成する能力が評価する質問。質問に直接知識がないため、事実を確認しないため、最新情報や組織に固有の知識に似たような質問で評価が低下する。
8,言い換え,APIを利用する人を識別する秘密の文字列を何と呼びますか？,APIキー,API を利用する人を識別する秘密の文字列を**API キー**（あるいは API Key）と呼びます。\n\nこれは、特定のサービス（例：Google API、AWS API、Slack API など）にアクセスするための一意のパスワードや認証コードのことです。\n\n* **API キー**：サービスにアクセスするための認証情報。\n* **API キー**：サービスにアクセスするための認証情報。\n\nこの情報は、ユーザーのパスワード,MCPサーバーに書くための秘密の文字列。APIキーのような秘密情報や、サーバーの構造を公開しない情報（ヘッダーのURL）を、MCPクライアントが読み込むときだけ公開する設計。本講座ではAPIキーを題材に、そのキーを公開するとクライアントが複数利用しやすくなる。
9,言い換え,LLMが1回で扱える入力と出力の合計トークン数の上限を何と呼びますか？,コンテキストウィンドウ,LLM（大規模言語モデル）が 1 回で処理できる入力と出力の合計トークン数の上限を**「最大トークン数（Maximum Token Length）」**と呼びます。\n\nこの値は、モデルの設計パラメータ（例えば、GPT-3.5 の 4096 個のトークン、GPT-4 の 8192 個など）によって異なります。また、この上限は,APIの料金やレート制限で上限に達しないように設ける設定値。より多くの質問を返すため、回答の書き方や形式に絞ってトークン数を減らす必要がある。


## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [9]:

# 単純な用語名検索
def glossary_search(question):
    for term in sorted(GLOSSARY, key=len, reverse=True):
        if term in question:
            return GLOSSARY[term]

    return "用語集に記載なし"

search_rows = []

for ex in test:
    answer = glossary_search(ex["question"])
    search_rows.append({
        "question": ex["question"],
        "raw": answer
    })

# 学習前・学習後・検索方式の3つを比較
comparison = pd.DataFrame([
    {
        "区分": ex["kind"],
        "質問": ex["question"],
        "正解": (
            gold[ex["question"]]["expected"]
            if gold[ex["question"]]["expected"] is not None
            else "用語集に記載なし"
        ),
        "学習前": b["raw"],
        "学習後": a["raw"],
        "検索方式": s["raw"]
    }
    for ex, b, a, s in zip(
        test, before_rows, after_rows, search_rows
    )
])

display(comparison)

comparison.to_csv(
    WORK / "comparison.csv",
    index=False,
    encoding="utf-8-sig"
)


,区分,質問,正解,学習前,学習後,検索方式
0,収録語,「LoRA」とは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
1,収録語,「過学習」とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
2,収録語,「ハルシネーション」とは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,**ハルシネーション**（Hārinēshō）は、日本語で「**ハルシネーション**」です。\n\nこの言葉は、主に**物理学**（特に光の現象）や**音楽学**（特に弦楽器の音）において使われます。\n\n* **物理学の定義**: 光の波が媒質（空気、水など）を通過する際、波の振幅が媒質の密度や,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
3,収録語,「APIキー」とは何ですか？,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。,**API キー**（Application API Key）とは、アプリケーション（アプリやサービス）が、外部のサービス（API）にアクセスするための**認証情報**です。\n\n簡単に言うと、「あなたのアプリが誰かにデータを送るためのパスワード」や「アクセス権限の証明」のことです。\n\n### 主な特徴と仕組み\n\n1. **認証の必須条件**\n 多くの API サービス（例：Google API、Facebook API、AWS API など,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。
4,収録語,「コンテキストウィンドウ」とは何ですか？,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,**コンテキストウィンドウ**（Context Window）とは、大規模なテキストベースの AI（特に大規模言語モデル）において、**「入力されたテキストの全体」を一つの大きな情報セットとして認識・処理する仕組み**です。\n\n従来のテキスト処理では、単語単位で処理する一方、コンテキストウィンドウでは、**「文脈（前後の文脈）」を考慮して、単語の意味や文脈を同時に理解・生成する,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。
5,言い換え,元の重みを固定し、低ランク行列だけを学習する手法は何ですか？,LoRA,元の重みを固定し、低ランク行列だけを学習する手法は、**低ランク学習（Low-Rank Learning）**、あるいは**低ランク分解（Low-Rank Decomposition）**と呼ばれます。\n\nこの手法は、データやモデルの構造を「低ランク」であると仮定して、その構造を最小二乗法などで最適化することで、より効率的に学習できるアプローチです。\n\n具体的には、以下の 2 つの主要なアプローチがあります。,PNN（Proximal Neuron）やLoRA。LoRAでは重みを交換しないので学習が安定し、PNNでは初期の好な行列を共有するので学習も安定する。,用語集に記載なし
6,言い換え,訓練データには適合するのに未知のデータで性能が下がる状態を何と呼びますか？,過学習,その状態を**欠損値（Missing Value）**と呼びます。\n\nこれは、訓練データに含まれていない（または入力された値が不明）部分のことを指します。この欠損値が訓練データに直結している場合、モデルは欠損値を無視して学習し、結果として**欠損率（Missing Rate）**が高くなることで、最終的な予測精度が低下します。\n\n欠損値の処理には、**欠損値,未知の入力に直面し、モデルの挙動や出力が安定しないようにデータに付けている手法の総称。Pydantic AIでは **LoRAアダプタの lora LoRA居士**。モデルが知らない入力に直面し、モデルが知らないことにもっとする精度落ちを補うための保険の型。本講座第2回の演習ではLoRAアダプタを扱う。,用語集に記載なし
7,言い換え,LLMが事実と異なる情報をもっともらしく生成する現象は何ですか？,ハルシネーション,LLM（大規模言語モデル）が事実と異なる情報を「もっともらしく」生成する現象は、主に**「誤解された事実（Misinformation）」**や**「誤った情報（Misleading Information）」**と呼ばれます。\n\nこれは、モデルが学習した知識に基づいて、特定の文脈や目的で歪曲された情報を生成する現象です。その特徴は以下の通りです。\n\n1. **事実と異なる情報**：モデルが,質問に存在する事実と異なる情報を、モデルが知らないことや、訓練データにないことや、最新情報や組織に固有の情報や、特定の業界の常識や教条に似たような情報を、もっともらしく生成する能力が評価する質問。質問に直接知識がないため、事実を確認しないため、最新情報や組織に固有の知識に似たような質問で評価が低下する。,用語集に記載なし
8,言い換え,APIを利用する人を識別する秘密の文字列を何と呼びますか？,APIキー,API を利用する人を識別する秘密の文字列を**API キー**（あるいは API Key）と呼びます。\n\nこれは、特定のサービス（例：Google API、AWS API、Slack API など）にアクセスするための一意のパスワードや認証コードのことです。\n\n* **API キー**：サービスにアクセスするための認証情報。\n* **API キー**：サービスにアクセスするための認証情報

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [11]:

# 最後に記録されたTraining Loss
loss_logs = [
    log for log in trainer.state.log_history
    if "loss" in log
]

last_loss = loss_logs[-1]["loss"] if loss_logs else None

# 学習全体の記録
train_logs = [
    log for log in trainer.state.log_history
    if "train_loss" in log
]

train_metrics = train_logs[-1] if train_logs else {}

result = {
    "config": {
        k: v for k, v in CONFIG.items()
        if k not in ("work_dir", "drive_dir")
    },

    "versions": {
        m.__name__: m.__version__
        for m in [transformers, peft, trl, torch]
    },

    "method": "LoRA（量子化なし）/ 用語集SFT",

    "train_examples": len(records),
    "glossary_terms": len(GLOSSARY),
    "test_examples": len(test),

    "before": before_summary,
    "after": after_summary,

    "manual_scores": globals().get("manual_scores", None),

    "last_training_loss": last_loss,
    "train_loss": train_metrics.get("train_loss"),
    "training_seconds": train_metrics.get("train_runtime"),
    "training_steps": trainer.state.global_step,

    "executed_training_myself": True
}

(WORK / "result.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(result, ensure_ascii=False, indent=2))

# Google Driveに保存する場合
if USE_DRIVE:
    import shutil

    for name in [
        "result.json",
        "eval_before.json",
        "eval_after.json",
        "test.jsonl",
        "gold.json",
        "comparison.csv"
    ]:
        shutil.copy(
            WORK / name,
            Path(CONFIG["drive_dir"]) / name
        )


{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 4,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）/ 用語集SFT",
  "train_examples": 116,
  "glossary_terms": 58,
  "test_examples": 13,
  "before": {
    "label": "before",
    "n": 13,
    "seconds": 89.0
  },
  "after": {
    "label": "after",
    "n": 13,
    "seconds": 46.6
  },
  "manual_scores": null,
  "last_training_loss": 0.06873367428779602,
  "train_loss": 0.3740508250395457,
  "training_seconds": 681.7784,
  "training_steps": 300,
  "executed_training_myself": true
}


## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。良くなる方向に変わる質問もあれば、`needs_review` が `"false"`（文字列）になって型検査で落ちる、根拠なしの質問に「はい」「なし」と断定する、といった出力も見られます。方向が一定でないことを確かめてください。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))